# GitHub Copilot Workshop · Microsoft Fabric

GitHub Copilot is the coding assistant; Fabric is the execution target. Synthetic manufacturing data only. See `docs/notebooks.md` before running. Use a disposable sandbox: data notebooks perform guarded full refreshes, not production ingestion.


In [ ]:
workspace_id = ""
lakehouse_id = ""
sandbox = False
approval = ""
batch_id = ""
expected_model_id = ""
model_workspace_id = ""
apply_changes = False
apply_approval = ""
reviewed_diff_hash = ""
selected_changes = []
verify_model = False


In [ ]:
"""Fail-closed Fabric notebook authorization, also embedded by the builder."""

import re
from uuid import UUID


def require_uuid(value: str, label: str) -> str:
    if not isinstance(value, str):
        raise ValueError(f"{label} must be an explicit canonical UUID")
    try:
        parsed = UUID(value)
    except (ValueError, AttributeError) as error:
        raise ValueError(f"{label} must be an explicit canonical UUID") from error
    if str(parsed) != value or parsed.int == 0:
        raise ValueError(f"{label} must be a nonzero lowercase canonical UUID")
    return value


def assert_sandbox_context(runtime_context, *, workspace_id: str, lakehouse_id: str,
                           sandbox: bool, approval: str, batch_id: str) -> None:
    """Names, implicit current-workspace defaults, and truthy strings are not consent."""
    require_uuid(workspace_id, "workspace_id")
    require_uuid(lakehouse_id, "lakehouse_id")
    if sandbox is not True or approval != workspace_id:
        raise ValueError("Require sandbox=True and approval equal to the exact workspace UUID")
    if not isinstance(batch_id, str) or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,79}", batch_id):
        raise ValueError("batch_id must be a nonempty, path-safe run identifier (at most 80 characters)")
    try:
        current_workspace = runtime_context["currentWorkspaceId"]
        default_lakehouse = runtime_context["defaultLakehouseId"]
    except (KeyError, TypeError, AttributeError) as error:
        raise ValueError("Fabric runtime workspace and default lakehouse UUIDs are required") from error
    if current_workspace != workspace_id or default_lakehouse != lakehouse_id:
        raise ValueError("Fabric runtime UUIDs do not match the explicitly approved sandbox")
    attached_workspace = runtime_context.get("defaultLakehouseWorkspaceId")
    if attached_workspace and attached_workspace != workspace_id:
        raise ValueError("Cross-workspace default lakehouses are not supported by this workshop")


def assert_model_target(*, workspace_id: str, model_workspace_id: str,
                        expected_model_id: str) -> None:
    require_uuid(model_workspace_id, "model_workspace_id")
    require_uuid(expected_model_id, "expected_model_id")
    if model_workspace_id != workspace_id:
        raise ValueError("The model must be in the explicitly approved sandbox workspace")

def require_sandbox():
    assert_sandbox_context(
        notebookutils.runtime.context, workspace_id=workspace_id, lakehouse_id=lakehouse_id,
        sandbox=sandbox, approval=approval, batch_id=batch_id,
    )

require_sandbox()


In [ ]:
"""Pure scoped model planning plus a small native TOM adapter; no network on import."""

import hashlib
import hmac
import json


def canonical_json(value) -> str:
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"))


def relationship_key(value) -> str:
    return ("relationship:" + value["from_table"] + "." + value["from_column"]
            + "->" + value["to_table"] + "." + value["to_column"])


def measure_key(value) -> str:
    return "measure:" + value["table"] + ":" + value["name"]


def snapshot_model(tom, workspace_id: str, model_id: str) -> dict:
    tables = []
    measures = []
    for table in tom.model.Tables:
        tables.append({"name": table.Name, "columns": sorted(column.Name for column in table.Columns)})
        for measure in table.Measures:
            measures.append({
                "table": table.Name, "name": measure.Name,
                "dax": measure.Expression or "", "format": measure.FormatString or "",
            })
    relationships = []
    for relation in tom.model.Relationships:
        if not hasattr(relation, "FromColumn"):
            raise ValueError("Unsupported relationship type; inspect and recover manually")
        relationships.append({
            "name": relation.Name,
            "from_table": relation.FromTable.Name, "from_column": relation.FromColumn.Name,
            "to_table": relation.ToTable.Name, "to_column": relation.ToColumn.Name,
            "active": bool(relation.IsActive),
            "from_cardinality": str(relation.FromCardinality),
            "to_cardinality": str(relation.ToCardinality),
            "cross_filtering_behavior": str(relation.CrossFilteringBehavior),
            "security_filtering_behavior": str(getattr(relation, "SecurityFilteringBehavior", "")),
            "rely_on_referential_integrity": bool(getattr(relation, "RelyOnReferentialIntegrity", False)),
        })
    snapshot = {
        "workspace_id": workspace_id, "model_id": model_id,
        "tables": sorted(tables, key=lambda value: value["name"]),
        "relationships": sorted(relationships, key=lambda value: value["name"]),
        "measures": sorted(measures, key=measure_key),
    }
    if hasattr(tom, "snapshot_definition"):
        # Native TOM serialization preserves annotations, partitions and other rollback context.
        snapshot["original_database_definition"] = tom.snapshot_definition()
    return snapshot


def make_plan(tom, contract: dict, workspace_id: str, model_id: str) -> dict:
    snapshot = snapshot_model(tom, workspace_id, model_id)
    tables = {table["name"]: set(table["columns"]) for table in snapshot["tables"]}
    missing = set(contract["tables"]) - set(tables)
    if missing:
        raise ValueError(f"Manually attach the missing core tables before planning: {sorted(missing)}")
    relationships = {}
    for relation in snapshot["relationships"]:
        key = relationship_key(relation)
        if key in relationships:
            raise ValueError("Duplicate relationship endpoints require manual review")
        relationships[key] = relation
    measures = {measure_key(measure): measure for measure in snapshot["measures"]}
    changes = []
    for desired in contract["relationships"]:
        for table, column in (("from_table", "from_column"), ("to_table", "to_column")):
            if desired[column] not in tables[desired[table]]:
                raise ValueError(f"Missing column {desired[table]}[{desired[column]}]; repair tables manually")
        key = relationship_key(desired)
        reverse = {
            "from_table": desired["to_table"], "from_column": desired["to_column"],
            "to_table": desired["from_table"], "to_column": desired["from_column"],
        }
        if relationship_key(reverse) in relationships:
            raise ValueError(f"A reversed relationship requires manual review: {key}")
        before = relationships.get(key)
        after = dict(desired, from_cardinality="Many", to_cardinality="One",
                     cross_filtering_behavior="OneDirection")
        if before is None or any(before[field] != after[field] for field in after):
            changes.append({"key": key, "kind": "relationship", "before": before, "after": after})
    for desired in contract["measures"]:
        key = measure_key(desired)
        before = measures.get(key)
        other_home = [measure for measure in snapshot["measures"]
                      if measure["name"] == desired["name"] and measure["table"] != desired["table"]]
        if other_home:
            raise ValueError(f"Measure [{desired['name']}] exists in a different home table; review manually")
        # Exact DAX comparison deliberately preserves string literal whitespace semantics.
        if before != desired:
            changes.append({"key": key, "kind": "measure", "before": before, "after": dict(desired)})
    payload = {"snapshot": snapshot, "desired_contract": contract, "changes": changes}
    return dict(payload, digest=hashlib.sha256(canonical_json(payload).encode("utf-8")).hexdigest())


def validate_review(plan: dict, reviewed_diff_hash: str, selected_changes: list[str]) -> None:
    if (not isinstance(reviewed_diff_hash, str) or len(reviewed_diff_hash) != 64
            or not hmac.compare_digest(reviewed_diff_hash, plan["digest"])):
        raise ValueError("Provide the exact SHA-256 digest of the independently reviewed snapshot/diff")
    payload = {key: plan[key] for key in ("snapshot", "desired_contract", "changes")}
    if hashlib.sha256(canonical_json(payload).encode("utf-8")).hexdigest() != plan["digest"]:
        raise ValueError("Snapshot/diff drift: the reviewed plan was modified")
    if (not isinstance(selected_changes, list) or not selected_changes
            or any(not isinstance(key, str) for key in selected_changes)
            or len(set(selected_changes)) != len(selected_changes)):
        raise ValueError("Explicitly select a nonempty list of unique change keys")
    known = {change["key"] for change in plan["changes"]}
    if set(selected_changes) - known:
        raise ValueError("Selected edits must be a subset of the reviewed named changes")


def apply_plan(tom, contract: dict, plan: dict, *, reviewed_diff_hash: str,
               selected_changes: list[str], tom_types) -> list[str]:
    """Mutate only approved objects in memory. The caller alone may explicitly SaveChanges."""
    validate_review(plan, reviewed_diff_hash, selected_changes)
    current = make_plan(tom, contract, plan["snapshot"]["workspace_id"], plan["snapshot"]["model_id"])
    if current["digest"] != plan["digest"]:
        raise ValueError("Model drift since review; obtain a new snapshot/diff and approval")
    chosen = [change for change in plan["changes"] if change["key"] in selected_changes]
    if tom_types is None and any(change["kind"] == "relationship" and change["before"] is not None
                                 for change in chosen):
        raise ValueError("TOM enum types must be supplied before editing existing relationships")
    tables = {table.Name: table for table in tom.model.Tables}
    relations = {relation.Name: relation for relation in tom.model.Relationships}
    for change in chosen:
        desired = change["after"]
        if change["kind"] == "measure":
            if change["before"] is None:
                tom.add_measure(table_name=desired["table"], measure_name=desired["name"],
                                expression=desired["dax"], format_string=desired["format"])
            else:
                measure = next(value for value in tables[desired["table"]].Measures
                               if value.Name == desired["name"])
                measure.Expression = desired["dax"]
                measure.FormatString = desired["format"]
        elif change["before"] is None:
            tom.add_relationship(
                from_table=desired["from_table"], from_column=desired["from_column"],
                to_table=desired["to_table"], to_column=desired["to_column"],
                from_cardinality="Many", to_cardinality="One",
                cross_filtering_behavior="OneDirection", is_active=True,
            )
        else:
            relation = relations[change["before"]["name"]]
            relation.IsActive = True
            relation.FromCardinality = tom_types.RelationshipEndCardinality.Many
            relation.ToCardinality = tom_types.RelationshipEndCardinality.One
            relation.CrossFilteringBehavior = tom_types.CrossFilteringBehavior.OneDirection
    return [change["key"] for change in chosen]


class NativeModelEditor:
    """No save-on-close wrapper or automatic model-wide lineage/DAX normalization."""

    def __init__(self, model, tom_types):
        self.model = model
        self.types = tom_types

    def snapshot_definition(self):
        return json.loads(str(self.types.JsonSerializer.SerializeDatabase(self.model.Database)))

    def add_measure(self, *, table_name, measure_name, expression, format_string):
        measure = self.types.Measure()
        measure.Name = measure_name
        measure.Expression = expression
        measure.FormatString = format_string
        self.model.Tables[table_name].Measures.Add(measure)

    def add_relationship(self, *, from_table, from_column, to_table, to_column,
                         from_cardinality, to_cardinality, cross_filtering_behavior, is_active):
        relation = self.types.SingleColumnRelationship()
        relation.Name = f"{from_table}_{from_column}__{to_table}_{to_column}"
        if any(value.Name == relation.Name for value in self.model.Relationships):
            raise ValueError("Proposed relationship name conflicts with another object; review manually")
        relation.FromColumn = self.model.Tables[from_table].Columns[from_column]
        relation.ToColumn = self.model.Tables[to_table].Columns[to_column]
        relation.FromCardinality = getattr(self.types.RelationshipEndCardinality, from_cardinality)
        relation.ToCardinality = getattr(self.types.RelationshipEndCardinality, to_cardinality)
        relation.CrossFilteringBehavior = getattr(self.types.CrossFilteringBehavior, cross_filtering_behavior)
        relation.IsActive = is_active
        self.model.Relationships.Add(relation)


In [ ]:
CHECK_MEASURES_DAX = 'EVALUATE\nROW (\n    "Planned Units", [Planned Units],\n    "Actual Units", [Actual Units],\n    "Scrap Units", [Scrap Units],\n    "Scrap Rate %", [Scrap Rate %],\n    "Production Attainment %", [Production Attainment %],\n    "Downtime Minutes", [Downtime Minutes],\n    "Inspected Units", [Inspected Units],\n    "Defect Count", [Defect Count],\n    "Defect Rate %", [Defect Rate %],\n    "Maintenance Events", [Maintenance Events],\n    "Preventive Maintenance %", [Preventive Maintenance %],\n    "Orders", [Orders],\n    "Total Revenue", [Total Revenue],\n    "Late Order %", [Late Order %]\n)\n'
MONTHLY_DAX = 'EVALUATE\nSUMMARIZECOLUMNS (\n    \'dim_date\'[YearMonth],\n    "Actual Units", [Actual Units]\n)\nORDER BY \'dim_date\'[YearMonth]\n'


# 07 · Review, then recover named semantic model objects

First obtain an owner-prestaged Direct Lake semantic model with the **12 core Gold tables** below, excluding
`agg_plant_month`. Use the explicit workspace and model UUIDs, never discovery by display name.
This notebook defaults to **read-only model access**: it saves the original definition and proposed diff
in the approved Lakehouse. A later run may apply only independently reviewed, individually named edits.
No tables, partitions, sources, roles or unrelated DAX are rebuilt.


In [ ]:
require_sandbox()
TABLES = ["dim_date", "dim_plant", "dim_line", "dim_product", "dim_shift", "dim_customer", "dim_defect_type",
          "dim_asset", "fact_production_run", "fact_quality_inspection", "fact_maintenance_event",
          "fact_customer_order"]

RELATIONSHIPS = [
    ("fact_production_run", "DateKey", "dim_date", "DateKey"),
    ("fact_production_run", "PlantKey", "dim_plant", "PlantKey"),
    ("fact_production_run", "LineKey", "dim_line", "LineKey"),
    ("fact_production_run", "ProductKey", "dim_product", "ProductKey"),
    ("fact_production_run", "ShiftKey", "dim_shift", "ShiftKey"),
    ("fact_quality_inspection", "DateKey", "dim_date", "DateKey"),
    ("fact_quality_inspection", "PlantKey", "dim_plant", "PlantKey"),
    ("fact_quality_inspection", "LineKey", "dim_line", "LineKey"),
    ("fact_quality_inspection", "ProductKey", "dim_product", "ProductKey"),
    ("fact_quality_inspection", "DefectTypeKey", "dim_defect_type", "DefectTypeKey"),
    ("fact_maintenance_event", "DateKey", "dim_date", "DateKey"),
    ("fact_maintenance_event", "PlantKey", "dim_plant", "PlantKey"),
    ("fact_maintenance_event", "LineKey", "dim_line", "LineKey"),
    ("fact_maintenance_event", "AssetKey", "dim_asset", "AssetKey"),
    ("fact_customer_order", "DateKey", "dim_date", "DateKey"),
    ("fact_customer_order", "PlantKey", "dim_plant", "PlantKey"),
    ("fact_customer_order", "ProductKey", "dim_product", "ProductKey"),
    ("fact_customer_order", "CustomerKey", "dim_customer", "CustomerKey"),
]

MEASURES = [
    ("fact_production_run", "Planned Units", "SUM ( 'fact_production_run'[PlannedUnits] )", "#,0"),
    ("fact_production_run", "Actual Units", "SUM ( 'fact_production_run'[ActualUnits] )", "#,0"),
    ("fact_production_run", "Scrap Units", "SUM ( 'fact_production_run'[ScrapUnits] )", "#,0"),
    ("fact_production_run", "Scrap Rate %", "DIVIDE ( [Scrap Units], [Actual Units] + [Scrap Units] )", "0.00%"),
    ("fact_production_run", "Production Attainment %", "DIVIDE ( [Actual Units], [Planned Units] )", "0.0%"),
    ("fact_production_run", "Downtime Minutes", "SUM ( 'fact_production_run'[DowntimeMinutes] )", "#,0"),
    ("fact_quality_inspection", "Inspected Units", "SUM ( 'fact_quality_inspection'[InspectedUnits] )", "#,0"),
    ("fact_quality_inspection", "Defect Count", "SUM ( 'fact_quality_inspection'[DefectCount] )", "#,0"),
    ("fact_quality_inspection", "Defect Rate %", "DIVIDE ( [Defect Count], [Inspected Units] )", "0.00%"),
    ("fact_maintenance_event", "Maintenance Events", "COUNTROWS ( 'fact_maintenance_event' )", "#,0"),
    ("fact_maintenance_event", "Preventive Maintenance %",
     "DIVIDE ( CALCULATE ( [Maintenance Events], 'fact_maintenance_event'[PreventiveFlag] = 1 ), [Maintenance Events] )",
     "0.0%"),
    ("fact_customer_order", "Orders", "COUNTROWS ( 'fact_customer_order' )", "#,0"),
    ("fact_customer_order", "Total Revenue", "SUM ( 'fact_customer_order'[Revenue] )", "#,0"),
    ("fact_customer_order", "Late Order %", "DIVIDE ( SUM ( 'fact_customer_order'[LateShipmentFlag] ), [Orders] )", "0.0%"),
]

CONTRACT = {
    "version": 1, "tables": TABLES,
    "relationships": [dict(zip(("from_table", "from_column", "to_table", "to_column"), value, strict=True), active=True)
                      for value in RELATIONSHIPS],
    "measures": [dict(zip(("table", "name", "dax", "format"), value, strict=True)) for value in MEASURES],
}


## 1 · Snapshot and proposed diff (no model edits)

The shared guard executes before this cell. Workspace, Lakehouse and model IDs must all be explicit UUIDs.
Semantic Link is provided by supported Fabric runtimes; use a reviewed Fabric Environment if it is missing.
There is no automatic install, provisioning, refresh, model selection or authentication setup here.
Planning is model-read-only, not storage-read-only: the explicitly approved sandbox receives the snapshot.
Each Lakehouse directory/file write rechecks runtime UUIDs and the exact workspace approval.


In [ ]:
require_sandbox()
def require_model_context():
    require_sandbox()
    assert_model_target(workspace_id=workspace_id, model_workspace_id=model_workspace_id,
                        expected_model_id=expected_model_id)
    if type(apply_changes) is not bool or type(verify_model) is not bool:
        raise ValueError("apply_changes and verify_model must be Boolean values, not strings")


require_model_context()
from math import isfinite

import sempy.fabric as fabric

# Cache invalidation has only a workspace argument in the documented SemPy API.
fabric.refresh_tom_cache(workspace=model_workspace_id)
server = fabric.create_tom_server(dataset=expected_model_id, workspace=model_workspace_id, readonly=True)
try:
    import Microsoft.AnalysisServices.Tabular as TOM

    model = server.Databases[expected_model_id].Model
    if str(model.Database.ID) != expected_model_id:
        raise ValueError("Resolved model UUID differs from expected_model_id")
    editor = NativeModelEditor(model, TOM)
    plan = make_plan(editor, CONTRACT, model_workspace_id, expected_model_id)
finally:
    server.Disconnect()

require_model_context()
snapshot_folder = f"Files/workshop/recovery/{batch_id}/{plan['digest']}"
snapshot_path = f"{snapshot_folder}/plan.json"
snapshot_text = canonical_json(plan)
require_model_context()
notebookutils.fs.mkdirs(snapshot_folder)
if notebookutils.fs.exists(snapshot_path):
    existing_text = notebookutils.fs.head(snapshot_path, len(snapshot_text.encode("utf-8")) + 1)
    if existing_text != snapshot_text:
        raise ValueError("Existing rollback snapshot differs; use a new batch_id and investigate")
else:
    require_model_context()
    if not notebookutils.fs.put(snapshot_path, snapshot_text, False):
        raise ValueError("Could not preserve the original definition; no edits are permitted")
print("Snapshot and rollback definitions:", snapshot_path)
print("REVIEWED DIFF HASH (independent review required):", plan["digest"])
print(json.dumps(plan["changes"], indent=2))


## 2 · Separately approved, explicitly scoped apply

Default `apply_changes=False`. After downloading/reviewing the original definition and diff, supply the
displayed SHA-256 as `reviewed_diff_hash`, set `apply_approval` to the exact model UUID, list only the approved
`selected_changes` keys, and set `apply_changes=True`. A changed model or changed contract invalidates review.
Existing reversed relationships, wrong home tables and missing columns require manual remediation.
The client never saves in a `finally` block. On failure it disconnects without saving in-memory changes.


In [ ]:
require_sandbox()
require_model_context()
if apply_changes:
    if apply_approval != expected_model_id:
        raise ValueError("Separate apply_approval must equal the exact expected model UUID")
    if (plan["snapshot"]["workspace_id"], plan["snapshot"]["model_id"]) != (
            model_workspace_id, expected_model_id):
        raise ValueError("The reviewed plan targets a different model")
    validate_review(plan, reviewed_diff_hash, selected_changes)
    # Re-read server state instead of applying against a cached or stale client model.
    fabric.refresh_tom_cache(workspace=model_workspace_id)
    server = fabric.create_tom_server(dataset=expected_model_id, workspace=model_workspace_id, readonly=False)
    try:
        live_model = server.Databases[expected_model_id].Model
        if str(live_model.Database.ID) != expected_model_id:
            raise ValueError("Writable model UUID differs from expected_model_id")
        live_editor = NativeModelEditor(live_model, TOM)
        applied = apply_plan(live_editor, CONTRACT, plan, reviewed_diff_hash=reviewed_diff_hash,
                             selected_changes=selected_changes, tom_types=TOM)
        require_model_context()
        live_model.SaveChanges()
        print("Saved only these approved changes:", applied)
    finally:
        server.Disconnect()
    fabric.refresh_tom_cache(workspace=model_workspace_id)
else:
    print("Read-only model plan complete. No semantic model changes were saved.")


## 3 · Optional read-only verification after model completion

Set `verify_model=True` only when all required measures/relationships exist. Querying does not refresh data.
If Direct Lake metadata needs refresh, perform that manually in the Fabric UI with appropriate permission.
Keep the saved snapshot for manual scoped restoration; never deploy the entire old database over new work.


In [ ]:
require_sandbox()
require_model_context()
if verify_model:
    result = fabric.evaluate_dax(dataset=expected_model_id, dax_string=CHECK_MEASURES_DAX,
                                 workspace=model_workspace_id)
    display(result)
    row = result.iloc[0].tolist()
    expected = [
        ("Planned Units", 1095550, 0), ("Actual Units", 1032704, 0), ("Scrap Units", 33404, 0),
        ("Scrap Rate %", 0.0313, 0.00005), ("Production Attainment %", 0.9426, 0.00005),
    ]
    for (name, value, tolerance), actual in zip(expected, row[:5], strict=True):
        if actual is None or not isfinite(float(actual)) or abs(float(actual) - value) > tolerance:
            raise ValueError(f"{name}: expected {value}, got {actual}")
    monthly = fabric.evaluate_dax(dataset=expected_model_id, dax_string=MONTHLY_DAX,
                                  workspace=model_workspace_id)
    display(monthly)
    by_plant = fabric.evaluate_dax(
        dataset=expected_model_id, workspace=model_workspace_id,
        dax_string="""EVALUATE SUMMARIZECOLUMNS (
            'dim_plant'[PlantCode], "Scrap", ROUND ( [Scrap Rate %], 4 )
        ) ORDER BY 'dim_plant'[PlantCode]""",
    )
    scrap = {key: round(float(value), 4)
             for key, value in zip(by_plant.iloc[:, 0], by_plant.iloc[:, 1], strict=True)}
    if scrap != {"AU-01": 0.0294, "IN-01": 0.0292, "MY-01": 0.0379, "SG-01": 0.0291}:
        raise ValueError("Plant-filtered scrap rates differ; inspect the reviewed relationships")
    print("Headline values and plant relationships verified. Stop the Spark session.")
